# [The StatQuest Illustrated Guide to Machine Learning](https://www.amazon.com/StatQuest-Illustrated-Guide-Machine-Learning/dp/B09ZCKR4H6)

# Chapter 10 - Decision Trees

Copyright 2026, Joshua Starmer

In this notebooke we'll...

- Load data from a file into a **DataFrame**
- Filter the data
- Split the data into **Training** and **Testing Datasets**
- Build **Decsion Trees**
- Draw **Decision Trees**
- Draw **Confusion Matrices**

----

# Load data from a file

The first thing we need to do is get our data.

However, before we get to that, we have to load in some modules that will help us load data and build and display a decision tree and a confusion matrix.

In [ ]:
## NOTE: cuML is NVIDIA's library to speed up sklearn.
## Basically it allows us to run sklearn on the GPU without
## having to change our code.
## NOTE: the "cu" is short of CUDA, which is NVIDIA's 
## platform for accelerating things with a GPU
##
## cuML is preinstalled on Google Colab - so we can not use it 
## and then see the speed up when using it. bam.
##
#%load_ext cuml.accel

import pandas as pd # to load and manipulate data and for One-Hot Encoding
import numpy as np # to calculate the mean and standard deviation
import matplotlib.pyplot as plt # to draw graphs
from sklearn.tree import DecisionTreeClassifier # to build a classification tree
from sklearn.tree import plot_tree # to draw a classification tree
from sklearn.model_selection import train_test_split # to split data into training and testing sets
from sklearn.model_selection import cross_val_score # for cross validation
from sklearn.metrics import ConfusionMatrixDisplay # creates and draws a confusion matrix

Now we can load in data. We'll use the Pandsa function `read_csv()`, where we need to specify the name of the file (and the path to the file, if necessary), and the character used to separate the columns of data. In this case, `geochem_data.txt` is a comma-delimited file so we'll set `sep='\t'`.

In [ ]:
## First, use pd.read_csv() to read the data in "geochem_data.txt"
geochem_data = pd.read_csv("https://raw.githubusercontent.com/StatQuest/sigml/refs/heads/main/geochem_data.txt", sep="\t")

# Verify that read_csv() was successful by printing out the first few rows
geochem_data.head()

Now let's see how many rows of data we have...

In [ ]:
## print out the number of rows and columns...
geochem_data.shape ## returns (# rows, # columns)

Now, the idea is to try to predict the value in the `LITH` column. So, let's first print out the different values in that column. We'll do this with the `unique()` DataFrame method, which is similar to the `set()` function, and omits all of the duplicate values and just returns the unique values.

In [ ]:
geochem_data['LITH'].unique()

For compairson, let's see what the `set()` function returns...

In [ ]:
set(geochem_data['LITH'])

Anyway, we see that there are 11 different values that `LITH` can have. Ideally, all 11 values would have an equal number of rows in the dataset. So, ideally, we would want each value to have 499/11 = 45 (rounded) rows. However, we rarely have such balanced datasets, so lets count the number of rows for each value and see what we have.

In [ ]:
# how many rows in each category for 'LITH'?
counts = []

for label in geochem_data['LITH'].unique():
    count = sum(geochem_data['LITH'] == label)
        
    counts.append(count)

count_data = {
    'label': geochem_data['LITH'].unique(),
    'counts': counts
}
count_data = pd.DataFrame(count_data)
count_data

So, instead of 45 rows, some of the categories have hardly any data and that means trouble. First, for the categories with only one row associated with it, we can either train a tree to predict it, or we can test a tree to predict it, but we can't do both. So that's a problem. The second problem is that when we only have a few rows of data for a category we will probably get a tree that is overfit for that category and won't make a good predictor. Alternatively, the tree could just completely ignore the categories with just a few rows and focus on everything else. So, in this example, we'll remove the categories that have less than 10 rows from the dataset.

In [ ]:
for label in geochem_data['LITH'].unique():
    count = sum(geochem_data['LITH'] == label)
    if (count < 10):
        # print("label:", label)
        # print("count:", count)
        geochem_data = geochem_data[geochem_data['LITH'] != label]

Now print out the categories and the number of rows associated with them to make sure we did things correctly.

In [ ]:
# how many rows in each category for 'LITH'?
counts = []

for label in geochem_data['LITH'].unique():
    count = sum(geochem_data['LITH'] == label)
        
    counts.append(count)

count_data = {
    'label': geochem_data['LITH'].unique(),
    'counts': counts
}
count_data = pd.DataFrame(count_data)
count_data

# BAM!

We're down to just 6 different values for `LITH`. Now let's check to see how many rows remain in the dataset...

In [ ]:
## print out the number of rows and columns...
geochem_data.shape ## returns (# rows, # columns)

...and we see that we still have most of the data we started with. When we elimited 5 values from `LITH`, we only remvoed 15 rows.

Now let's format the data for training and testing a machine learning model.

----

<a id="format-the-data"></a>
# Split the Data into Targets (Dependent Variables) and Features (Independent Variables)

Now removed data for underepresented categories, we are ready to start formatting the data for making a **Classification Tree**.

The first step is to split the data into two parts:
1. The columns of data that we will use to make classifications
2. The column of data that we want to predict.

We will use the conventional notation of `X` (capital **X**) to represent the columns of data that we will use to make classifications and `y` (lower case **y**) to represent the thing we want to predict. In this case, we want to predict `LITH`.

**NOTE:** In the code below we are using `copy()` to copy the data *by value*. By default, pandas uses copy *by reference*. Using `copy()` ensures that the original data `geochem_data` is not modified when we modify `X` or `y`. In other words, if we make a mistake when we are formatting the columns for classification trees, we can just re-copy `geochem_data`, rather than reload the original data and remove the missing values etc.

In [ ]:
## To create a DataFrame that just has the features, the
## columns we want to use to make predictions (also called
## the indpendent variables) we make a copy of the 
## dataset but "drop" the column we want to predict.
X = geochem_data.drop('LITH', axis=1).copy()
## print the first few rows
X.head()

In [ ]:
## To create a DataFrame that just has the target, the column
## we want to predict (also called the dependent variable), 
## just make a new copy of the column of data we want to predict
y = geochem_data['LITH'].copy()
## print the first few rows
y.head()

In [ ]:
## Now verify that 'y' contains the values we want
## to predict...
y.unique()

Now we're going to do something that might seem a little strange. We're going to make a new DataFrame that simplifies the classification to just 2 categories. We'll try to separarte rows of data with the `A` value from all of the other rows. This is sometimes called **1 vs. All** classification. Doing this ensures we just a binary classification problem, which, when it comes time to draw a **Confusion Matrix**, will be much easier to understand. Once we have a grasp of how binary classification works, we will return to using all of the values for `LITH` for classification.

In [ ]:
## For A vs All (binary classification)
## we start by making a new copy of the LITH column...
y_A_vs_all = geochem_data['LITH'].copy()
y_A_vs_all.head()

In [ ]:
## ...then we replace all of the non 'A' values
## with 'nA', which stands for "not A" (I thought
## about using "NA" for "not A", but "NA" usually
## stands for "not applicable", and thus, might 
## confuse people.
y_not_A_index = y_A_vs_all != 'A' # get the index for each non 'A' value in y
y_A_vs_all[y_not_A_index] = 'nA'  # set each non 'A' value in y to 'nA', which is "not A"
y_A_vs_all.unique()               # verify that y only contains 'A' and 'nA'.

Now that we have reduced the problem to a binary classification problem with `y_A_vs_All`, we can create the training and testing datasets, and the build a decision tree with the training data.

-----

# Splitting the data into training and testing datasets for binary classification...

In [ ]:
## split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y_A_vs_all, 
                                                    stratify=y_A_vs_all, ## try to maintain the same proportions for each value
                                                    random_state=42)

-----

# Building and testing a binary classification tree...

In [ ]:
## create a decisiont tree and fit it to the training data
clf_dt = DecisionTreeClassifier(random_state=42)
clf_dt = clf_dt.fit(X_train, y_train)

Now let's draw the tree that we built.

In [ ]:
## NOTE: We can plot the tree and it is huge!
plt.figure(figsize=(15, 7.5))
plot_tree(clf_dt, 
          filled=True, 
          rounded=True,
          feature_names=X.columns); 

Now let's see how how well our classification tree performs by running the testing data down it and using the results to draw a **Confusion Matrix**.

In [ ]:
## plot_confusion_matrix() will run the test data down the tree and draw
## a confusion matrix in one gow
ConfusionMatrixDisplay.from_estimator(clf_dt, 
                                      X_test, 
                                      y_test, 
                                      display_labels=y_test.unique())

As we can see in the **Confusion Matrix**, for the most part, our classification tree did a good job. Most of the `A` and `nA` values rows in the testing dataset were correctly classified.

# BAM!

Now do the same thing with all the target labels (with more than 10 rows of data)

-----

# Splitting the data into training and testing datasets for multiclass classification

First, let's remember which classes we want to predict...

In [ ]:
y.unique()

Now let's split that data into training and testing datasets...

In [ ]:
## split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

----

# Building and testing a multiclass classification tree...

In [ ]:
## create a decisiont tree and fit it to the training data
clf_dt = DecisionTreeClassifier(random_state=42)
clf_dt = clf_dt.fit(X_train, y_train)

Now let's draw the tree that we built.

In [ ]:
## NOTE: We can plot the tree and it is huge!
plt.figure(figsize=(15, 7.5))
plot_tree(clf_dt, 
          filled=True, 
          rounded=True,
          feature_names=X.columns); 

Now let's see how how well our classification tree performs by running the testing data down it and using the results to draw a **Confusion Matrix**.

In [ ]:
## plot_confusion_matrix() will run the test data down the tree and draw
## a confusion matrix.
ConfusionMatrixDisplay.from_estimator(clf_dt, 
                                      X_test, 
                                      y_test, 
                                      display_labels=y_test.unique())

A multiclass **Confusion Matrix** is a lot harder to read, but if we focus on the diagonal, we can see that, over all, our predictions are pretty good!

# TRIPLE BAM!